In [1]:
!pip install faker pandas pyarrow duckdb apache-airflow

In [2]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import sys
import os

# Hogy elérjük a scripts mappát
sys.path.append(os.path.join(os.path.dirname(__file__), 'scripts'))
from generator import generate_fake_data
from transform import run_transformations

default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'start_date': datetime(2024, 1, 1),
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

with DAG(
    'gas_iot_pipeline',
    default_args=default_args,
    description='IoT Gas Meter ETL pipeline',
    schedule_interval='@daily', 
    catchup=False
) as dag:

    # 1. Task: Adatgenerálás
    t1 = PythonOperator(
        task_id='generate_iot_data',
        python_callable=generate_fake_data,
    )

    # 2. Task: Transzformáció és betöltés a tárházba
    t2 = PythonOperator(
        task_id='transform_and_load_duckdb',
        python_callable=run_transformations,
    )

    t1 >> t2

ModuleNotFoundError: No module named 'data_functions'